# Tutorial 8 — Non-Degenerate OPA in PPLN

A **realistic, non-degenerate, single-pass optical parametric amplifier**
for mid-infrared generation.

**Use case:** Generate tunable mid-IR light at 3.6 um for C-H stretch
spectroscopy. Start with a 1.064 um Nd:YAG pump and a 1.507 um
erbium-fiber seed. Energy conservation produces the idler at 3.62 um.

**Contrast with Tutorial 7:** The degenerate OPO produced octave-spanning
broadband output with wild temporal modulation. This non-degenerate OPA
produces *clean, orderly output* — well-defined spectral peaks, smooth
temporal profiles. Same chi(2) physics, radically different character.

**Progression:**
1. Phase-matching landscape (QPM tuning curves)
2. Single-pass OPA — low gain
3. Single-pass OPA — high gain / saturation
4. Group velocity mismatch effects
5. Temperature tuning

In [ ]:
import numpy as np
from numpy.fft import fft, ifft, fftshift, fftfreq
import matplotlib.pyplot as plt
import time

import snow.pulses as pulses
import snow.materials as materials
import snow.waveguides as waveguides

from scipy.constants import pi, c
nm = 1e-9
um = 1e-6
mm = 1e-3
ps = 1e-12
fs = 1e-15
MHz = 1e6
THz = 1e12
pJ = 1e-12
nJ = 1e-9
uJ = 1e-6
uW = 1e-6
mW = 1e-3

plt.rcParams.update({'font.size': 14})

# Backend: 'jax' recommended — N=2^14 grid makes CPU very slow.
# A single pass at N=2^14 takes ~5s on GPU vs ~15 minutes on CPU.
BACKEND = 'jax'

nee_kwargs = {}
if BACKEND == 'jax':
    import jax.numpy as jnp

print(f'Backend: {BACKEND}')

## Wavelength geometry and grid

Energy conservation: 1/λ_p = 1/λ_s + 1/λ_i

| Wave   | Wavelength | Source                    |
|--------|------------|---------------------------|
| Pump   | 1.064 um   | Nd:YAG                   |
| Signal | 1.507 um   | Erbium fiber seed         |
| Idler  | 3.62 um    | Generated (mid-IR output) |

In [ ]:
# Wavelengths
lam_p = 1.064 * um   # pump
lam_s = 1.507 * um   # signal (seed)
lam_i = 1/(1/lam_p - 1/lam_s)  # idler from energy conservation
print(f'Pump:   {lam_p/um:.3f} um')
print(f'Signal: {lam_s/um:.3f} um')
print(f'Idler:  {lam_i/um:.4f} um  (energy conservation)')

# Grid: must span pump through idler with margin
lam_start = 900*nm
lam_stop = 4.2*um
f_max = c/lam_start
f_min = c/lam_stop
BW = f_max - f_min
N = 2**14  # Need large N for the wide bandwidth + 10ps pulses
dt = 1/BW
T_window = N * dt
t = -T_window/2 + np.arange(0, T_window, step=dt)
f = fftfreq(N, dt)
f_ref = c/lam_s  # reference frequency at signal wavelength
f_abs = f + f_ref
frep = 1*MHz  # low rep rate for high pulse energy

print(f'\nGrid: N={N}, BW={BW/THz:.1f} THz, dt={dt/ps:.4f} ps')
print(f'Time window: {T_window/ps:.1f} ps')
print(f'Wavelength range: {c/np.max(f_abs[f_abs>0])/um:.2f} - {c/np.min(f_abs[f_abs>0])/um:.2f} um')

## 1. Phase-matching landscape

Before any propagation, compute the QPM tuning curves.
For a given poling period and temperature, which signal/idler
pair satisfies both energy conservation and phase matching?

In [ ]:
T_design = 150  # design temperature (C)

# Compute QPM poling period for our target wavelengths
n_p = materials.refractive_index('LN_MgO_e_T', lam_p/um, T=T_design)
n_s = materials.refractive_index('LN_MgO_e_T', lam_s/um, T=T_design)
n_i = materials.refractive_index('LN_MgO_e_T', lam_i/um, T=T_design)
dk = n_p/lam_p - n_s/lam_s - n_i/lam_i  # phase mismatch per um
pp = 1/dk  # poling period
print(f'Design QPM poling period: {pp/um:.2f} um at T={T_design}C')
print(f'Refractive indices: n_p={n_p:.4f}, n_s={n_s:.4f}, n_i={n_i:.4f}')

# GVM
def group_index(lam_um, T):
    dl = 0.001
    n0 = materials.refractive_index('LN_MgO_e_T', lam_um, T=T)
    np_ = materials.refractive_index('LN_MgO_e_T', lam_um+dl, T=T)
    nm_ = materials.refractive_index('LN_MgO_e_T', lam_um-dl, T=T)
    return n0 - lam_um * (np_ - nm_)/(2*dl)

ng_p = group_index(lam_p/um, T_design)
ng_s = group_index(lam_s/um, T_design)
ng_i = group_index(lam_i/um, T_design)
gvm_ps = (ng_p - ng_s)/c * 1e9  # ps/mm
gvm_pi = (ng_p - ng_i)/c * 1e9
print(f'\nGroup indices: ng_p={ng_p:.4f}, ng_s={ng_s:.4f}, ng_i={ng_i:.4f}')
print(f'GVM pump-signal: {gvm_ps:.3f} ps/mm (walkoff over 20mm: {gvm_ps*20:.1f} ps)')
print(f'GVM pump-idler:  {gvm_pi:.3f} ps/mm (walkoff over 20mm: {gvm_pi*20:.1f} ps)')

In [ ]:
# Tuning curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), tight_layout=True)

# Plot A: signal/idler wavelength vs temperature (fixed poling period)
T_range = np.linspace(50, 250, 50)
lam_s_vs_T = []
lam_i_vs_T = []

for T_val in T_range:
    # For each T, find signal wavelength that satisfies QPM with our poling period
    # QPM: n_p/lam_p - n_s/lam_s - n_i/lam_i = 1/pp
    # with lam_i = 1/(1/lam_p - 1/lam_s)  (energy conservation)
    # Scan signal wavelength
    best_ls = lam_s
    best_dk = 1e10
    for ls_um in np.linspace(1.1, 2.0, 500):
        ls = ls_um * um
        li = 1/(1/lam_p - 1/ls)
        if li < 0: continue
        np_ = materials.refractive_index('LN_MgO_e_T', ls_um*um/um, T=T_val)
        ns_ = materials.refractive_index('LN_MgO_e_T', ls/um, T=T_val)
        ni_ = materials.refractive_index('LN_MgO_e_T', li/um, T=T_val)
        npp = materials.refractive_index('LN_MgO_e_T', lam_p/um, T=T_val)
        dk_val = abs(npp/lam_p - ns_/ls - ni_/li - 1/pp)
        if dk_val < best_dk:
            best_dk = dk_val
            best_ls = ls
    best_li = 1/(1/lam_p - 1/best_ls)
    lam_s_vs_T.append(best_ls/um)
    lam_i_vs_T.append(best_li/um)

ax1.plot(T_range, lam_s_vs_T, 'b-', label='Signal')
ax1.plot(T_range, lam_i_vs_T, 'r-', label='Idler')
ax1.axhline(y=lam_s/um, color='b', linestyle='--', alpha=0.3)
ax1.axhline(y=lam_i/um, color='r', linestyle='--', alpha=0.3)
ax1.axvline(x=T_design, color='k', linestyle=':', alpha=0.3)
ax1.set_xlabel('Temperature (C)'); ax1.set_ylabel('Wavelength (um)')
ax1.set_title(f'QPM Tuning Curve (pp={pp/um:.1f} um)');
ax1.legend(); ax1.grid(True)

# Plot B: signal/idler wavelength vs poling period (fixed T)
pp_range = np.linspace(25, 35, 50) * um
lam_s_vs_pp = []
lam_i_vs_pp = []

for pp_val in pp_range:
    best_ls = lam_s
    best_dk = 1e10
    for ls_um in np.linspace(1.1, 2.0, 500):
        ls = ls_um * um
        li = 1/(1/lam_p - 1/ls)
        if li < 0: continue
        npp = materials.refractive_index('LN_MgO_e_T', lam_p/um, T=T_design)
        ns_ = materials.refractive_index('LN_MgO_e_T', ls/um, T=T_design)
        ni_ = materials.refractive_index('LN_MgO_e_T', li/um, T=T_design)
        dk_val = abs(npp/lam_p - ns_/ls - ni_/li - 1/pp_val)
        if dk_val < best_dk:
            best_dk = dk_val
            best_ls = ls
    best_li = 1/(1/lam_p - 1/best_ls)
    lam_s_vs_pp.append(best_ls/um)
    lam_i_vs_pp.append(best_li/um)

ax2.plot(pp_range/um, lam_s_vs_pp, 'b-', label='Signal')
ax2.plot(pp_range/um, lam_i_vs_pp, 'r-', label='Idler')
ax2.axhline(y=lam_s/um, color='b', linestyle='--', alpha=0.3)
ax2.axhline(y=lam_i/um, color='r', linestyle='--', alpha=0.3)
ax2.axvline(x=pp/um, color='k', linestyle=':', alpha=0.3)
ax2.set_xlabel('Poling period (um)'); ax2.set_ylabel('Wavelength (um)')
ax2.set_title(f'QPM vs Poling Period (T={T_design}C)')
ax2.legend(); ax2.grid(True)

plt.show()

## PPLN crystal setup

We use the `waveguide` class in behavioral mode to set the dispersion
from the temperature-dependent Sellmeier model directly. This lets us
model a bulk PPLN crystal (no waveguide confinement needed — the
effective index IS the bulk index at these wavelengths).

In [ ]:
L_crystal = 20*mm
T_crystal = T_design  # 150C

# d33 = 27 pm/V for LiNbO3, deff = 2*d33/pi for first-order QPM
d33 = 27e-12  # m/V
d_eff = 2*d33/pi

# Build dispersion from material model
# We need n(lambda) for the extraordinary axis at temperature T
def n_func(wl):
    """Refractive index for e-polarized LN at T_crystal."""
    return materials.refractive_index('LN_MgO_e_T', wl, T=T_crystal)

# Compute beta(f) on our grid
wl_grid = c / f_abs  # wavelength at each frequency point
# Only evaluate where wavelength is positive and in valid range
valid = (f_abs > 0) & (wl_grid > 0.4*um) & (wl_grid < 5*um)
n_grid = np.ones(N)
n_grid[valid] = np.array([materials.refractive_index('LN_MgO_e_T', w/um, T=T_crystal)
                          for w in wl_grid[valid]])
beta_grid = 2*pi*f_abs * n_grid / c

# Reference velocity: use the signal group velocity
v_ref = c / ng_s

# Dispersion operator
Omega = 2*pi*f
beta_ref = 2*pi*f_ref * n_s / c
beta_1_ref = 1/v_ref
D = beta_grid - beta_ref - Omega/v_ref

# Nonlinear coupling
# For bulk crystal: k(z) = d_eff * omega_abs / (n * c) * poling(z)
# The NEE convention uses X0 / (4*N_mode)
# For bulk (plane wave): the coupling is d_eff * omega / (n*c)
# We need to match the NEE's internal convention
omega_abs = 2*pi*f_abs

# X0 parameter: snow's convention is k(z) = poling(z) * X0 * omega_abs / (4 * N_mode)
# Physical coupling for QPM: kappa = d_eff * omega / (n*c)
# So X0 / (4*1) should give d_eff / (n*c) * omega when multiplied by omega
# This means X0 = 4 * d_eff / (n_eff * c)... but it's frequency dependent.
# Actually, looking at the code: k(z) returns an array = poling(z) * X0 * omega_abs / (4*N)
# The actual coupling coefficient should be ~ d_eff * omega / (n * c)
# So we set X0 such that X0 * omega / 4 ≈ d_eff * omega / (n * c)
# => X0 = 4 * d_eff / (n_avg * c)
n_avg = np.mean([n_p, n_s, n_i])
X0 = 4 * d_eff / (n_avg * c)

def poling(z):
    return np.sign(np.cos(z * 2*pi / pp))

def k_func(z):
    return poling(z) * X0 * omega_abs / 4

if BACKEND == 'jax':
    poling_jax = lambda z: jnp.sign(jnp.cos(z * 2*jnp.pi / pp))
    nee_kwargs = {'poling_fn_jax': poling_jax}

print(f'Crystal: {L_crystal/mm:.0f} mm PPLN, T={T_crystal}C')
print(f'd_eff = {d_eff*1e12:.1f} pm/V, X0 = {X0:.3e}')
print(f'Poling period: {pp/um:.2f} um')

## 2. Single-pass OPA — low gain

Pump at 1.064 um (1 uJ), seed at 1.507 um (1 nJ).
Expect 10-30 dB signal gain, clean idler at 3.62 um.

In [ ]:
# Pump: 10 ps sech at 1.064 um, 1 uJ
E_pump = 1*uJ
tau = 10*ps
pump = pulses.sech_pulse(t, tau, f_ref=f_ref, f0=c/lam_p,
                         Energy=E_pump, Npwr_dB=200, frep=frep)

# Seed: 10 ps at 1.507 um, 1 nJ
E_seed = 1*nJ
seed = pulses.sech_pulse(t, tau, f_ref=f_ref, f0=c/lam_s,
                         Energy=E_seed, Npwr_dB=200, frep=frep)

input_opa = pump + seed
print(f'Pump: {E_pump/uJ:.0f} uJ at {lam_p/um:.3f} um, {tau/ps:.0f} ps')
print(f'Seed: {E_seed/nJ:.0f} nJ at {lam_s/um:.3f} um, {tau/ps:.0f} ps')

# Propagate
import snow.nlo_jax as nlo_jax
import snow.nlo_scipy as nlo_scipy

nee_base = dict(t=t, x=input_opa.a, Omega=Omega, f0=f_ref,
                L=L_crystal, D=D, b0=float(beta_ref), b1_ref=float(beta_1_ref),
                k=k_func, verbose=True)

t0 = time.perf_counter()
if BACKEND == 'jax':
    a_out, steps = nlo_jax.NEE(**nee_base, **nee_kwargs)
else:
    a_out, steps = nlo_scipy.NEE(**nee_base)
elapsed = time.perf_counter() - t0

out_pulse = pulses.pulse(t, a_out, c/f_ref, frep)
print(f'[{BACKEND}] {elapsed:.1f}s, {len(steps)} steps')

In [ ]:
# Analyze output
sig_out = out_pulse.apply_filter(c/lam_s, 20*THz)
idl_out = out_pulse.apply_filter(c/lam_i, 20*THz)
pump_out = out_pulse.apply_filter(c/lam_p, 20*THz)

gain_dB = 10*np.log10(sig_out.energy_td() / seed.energy_td())
pump_depl = 1 - pump_out.energy_td() / pump.energy_td()

print(f'Signal gain: {gain_dB:.1f} dB')
print(f'Idler energy: {idl_out.energy_td()/nJ:.3f} nJ')
print(f'Pump depletion: {pump_depl:.1%}')
print(f'Energy conservation: in={input_opa.energy_td()/uJ:.4f}, '
      f'out={out_pulse.energy_td()/uJ:.4f} uJ')

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), tight_layout=True)

# Spectra
input_opa.plot_PSD(ax=ax1, f_unit='um')
out_pulse.plot_PSD(ax=ax1, f_unit='um')
ax1.get_lines()[-2].set(color='k', linestyle='--', alpha=0.3, label='Input')
ax1.get_lines()[-1].set(color='b', linewidth=1.5, label='Output')
ax1.axvline(x=lam_i/um, color='r', linestyle=':', alpha=0.5, label=f'Idler ({lam_i/um:.2f} um)')
ax1.set_xlim(0.9, 4.2)
ax1.legend(); ax1.set_title('OPA Spectrum (low gain)')

# Temporal
input_opa.plot_magsq(ax=ax2, t_unit='ps')
out_pulse.plot_magsq(ax=ax2, t_unit='ps')
ax2.get_lines()[-2].set(color='k', linestyle='--', alpha=0.3, label='Input')
ax2.get_lines()[-1].set(color='b', linewidth=1.5, label='Output')
ax2.set_xlim(-20, 20)
ax2.legend(); ax2.set_title('Temporal Intensity')
plt.show()

### Analytical gain cross-check

This is the most important validation in the notebook.  The small-signal
gain for a perfectly phase-matched OPA in the undepleted-pump regime is:

G = cosh²(g·L),  where  g = sqrt(ω_s · ω_i · d_eff² · I_pump / (n_s · n_i · ε₀ · c³))

If the simulation gain doesn't match this to within a factor of ~2,
the field normalization (power vs intensity) is wrong.

**SNOW uses power-normalized fields:** |a|² = Watts.  For a bulk crystal
plane-wave simulation, we need to convert pump power to intensity via
the beam area A_eff = π·w₀².  If X0 was set without this factor, the
gain will be wrong.

In [ ]:
from scipy.constants import epsilon_0

# Beam parameters (not modeled by SNOW's 1D propagation, but needed for
# the analytical comparison and to set the correct coupling strength)
w0 = 100*um  # 1/e² beam waist radius
A_eff = pi * w0**2  # effective beam area

# Peak pump intensity
P_peak_pump = 0.88 * E_pump / tau  # sech peak power
I_pump = P_peak_pump / A_eff  # W/m²
print(f'Beam waist: {w0/um:.0f} um, A_eff = {A_eff*1e8:.1f} x 10^-4 cm²')
print(f'Peak pump power: {P_peak_pump:.0f} W')
print(f'Peak pump intensity: {I_pump/1e13:.2f} x 10^13 W/m² = {I_pump/1e9:.2f} GW/cm²')

# Analytical parametric gain coefficient
omega_s = 2*pi*c/lam_s
omega_i = 2*pi*c/lam_i
g_analytic = np.sqrt(omega_s * omega_i * d_eff**2 * I_pump / (n_s * n_i * epsilon_0 * c**3))
print(f'\nParametric gain coefficient g = {g_analytic:.2f} /m')
print(f'g·L = {g_analytic * L_crystal:.2f}')

G_analytic = np.cosh(g_analytic * L_crystal)**2
G_analytic_dB = 10*np.log10(G_analytic)
print(f'Analytical gain G = cosh²(gL) = {G_analytic:.1f}x = {G_analytic_dB:.1f} dB')

# Compare to simulation
print(f'\nSimulation gain: {gain_dB:.1f} dB')
print(f'Ratio (sim/analytical): {10**(gain_dB/10) / G_analytic:.2f}')

if abs(gain_dB - G_analytic_dB) > 20:
    print(f'\n*** LARGE DISCREPANCY ({abs(gain_dB - G_analytic_dB):.0f} dB) ***')
    print('The X0 coupling likely needs an A_eff correction.')
    print(f'Current X0 = {X0:.3e}')
    X0_corrected = X0 / np.sqrt(A_eff)
    print(f'Try X0 = X0 / sqrt(A_eff) = {X0_corrected:.3e}')
    print('Re-run the crystal setup cell with the corrected X0.')
elif abs(gain_dB - G_analytic_dB) < 6:
    print(f'\nGood agreement (within {abs(gain_dB - G_analytic_dB):.1f} dB).')
    print('Field normalization is correct for these parameters.')
else:
    print(f'\nModerate discrepancy ({abs(gain_dB - G_analytic_dB):.0f} dB).')
    print('May be due to GVM, pump depletion, or imperfect phase matching.')

## 3. Single-pass OPA — high gain / saturation

Increase pump energy to drive into saturation.
Expect back-conversion, pump depletion hole, spectral broadening.

In [ ]:
# Gain curve: sweep pump energy
E_pumps = np.array([0.1, 0.3, 1, 3, 5, 10]) * uJ
gains = []
idler_energies = []
depletions = []

print(f'{"E_pump":>8s} | {"Gain (dB)":>10s} | {"E_idler":>10s} | {"Depletion":>10s}')
print('-'*48)

for E_p in E_pumps:
    p = pulses.sech_pulse(t, tau, f_ref=f_ref, f0=c/lam_p,
                          Energy=E_p, Npwr_dB=200, frep=frep)
    inp = p + seed
    nee = dict(t=t, x=inp.a, Omega=Omega, f0=f_ref,
               L=L_crystal, D=D, b0=float(beta_ref), b1_ref=float(beta_1_ref),
               k=k_func, verbose=False)

    if BACKEND == 'jax':
        a, _ = nlo_jax.NEE(**nee, **nee_kwargs)
    else:
        a, _ = nlo_scipy.NEE(**nee)

    out = pulses.pulse(t, a, c/f_ref, frep)
    s_out = out.apply_filter(c/lam_s, 20*THz)
    i_out = out.apply_filter(c/lam_i, 20*THz)
    p_out = out.apply_filter(c/lam_p, 20*THz)

    g = 10*np.log10(s_out.energy_td() / seed.energy_td())
    gains.append(g)
    idler_energies.append(i_out.energy_td())
    depletions.append(1 - p_out.energy_td()/p.energy_td())

    print(f'{E_p/uJ:7.1f}uJ | {g:>9.1f} dB | {i_out.energy_td()/nJ:>8.2f} nJ | {depletions[-1]:>9.1%}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), tight_layout=True)
ax1.semilogx(E_pumps/uJ, gains, 'bo-')
ax1.set_xlabel('Pump energy (uJ)'); ax1.set_ylabel('Signal gain (dB)')
ax1.set_title('OPA Gain Curve'); ax1.grid(True)

ax2.loglog(E_pumps/uJ, np.array(idler_energies)/nJ, 'rs-', label='Idler')
ax2.set_xlabel('Pump energy (uJ)'); ax2.set_ylabel('Idler energy (nJ)')
ax2r = ax2.twinx()
ax2r.plot(E_pumps/uJ, np.array(depletions)*100, 'g^--', label='Pump depletion')
ax2r.set_ylabel('Pump depletion (%)')
ax2.set_title('Output Energy & Depletion'); ax2.grid(True)
ax2.legend(loc='upper left'); ax2r.legend(loc='center right')
plt.show()

In [ ]:
# Compare spectra and temporal profiles: low gain vs high gain
fig, axes = plt.subplots(2, 2, figsize=(14, 10), tight_layout=True)

for idx, E_p in enumerate([1*uJ, 10*uJ]):
    p = pulses.sech_pulse(t, tau, f_ref=f_ref, f0=c/lam_p,
                          Energy=E_p, Npwr_dB=200, frep=frep)
    inp = p + seed
    nee = dict(t=t, x=inp.a, Omega=Omega, f0=f_ref,
               L=L_crystal, D=D, b0=float(beta_ref), b1_ref=float(beta_1_ref),
               k=k_func, verbose=False)
    if BACKEND == 'jax':
        a, _ = nlo_jax.NEE(**nee, **nee_kwargs)
    else:
        a, _ = nlo_scipy.NEE(**nee)
    out = pulses.pulse(t, a, c/f_ref, frep)

    # Spectrum
    ax = axes[0, idx]
    inp.plot_PSD(ax=ax, f_unit='um')
    out.plot_PSD(ax=ax, f_unit='um')
    ax.get_lines()[-2].set(color='k', linestyle='--', alpha=0.3, label='Input')
    ax.get_lines()[-1].set(color='b', linewidth=1.5, label='Output')
    ax.set_xlim(0.9, 4.2)
    ax.legend(); ax.set_title(f'Spectrum — {E_p/uJ:.0f} uJ pump')

    # Temporal
    ax = axes[1, idx]
    inp.plot_magsq(ax=ax, t_unit='ps')
    out.plot_magsq(ax=ax, t_unit='ps')
    ax.get_lines()[-2].set(color='k', linestyle='--', alpha=0.3, label='Input')
    ax.get_lines()[-1].set(color='b', linewidth=1.5, label='Output')
    ax.set_xlim(-20, 20)
    ax.legend(); ax.set_title(f'Temporal — {E_p/uJ:.0f} uJ pump')

plt.show()

## 4. Group velocity mismatch effects

Compare different crystal lengths and pulse durations to see
how GVM shapes the output. With GVM ~ 0.09 ps/mm pump-signal,
the 10 ps pulses walk off by ~1.9 ps in 20 mm — modest but visible.
Shorter pulses or longer crystals make it dramatic.

In [ ]:
cases = [
    ('10 ps, 20 mm', 10*ps, 20*mm),
    ('10 ps, 5 mm',  10*ps, 5*mm),
    ('1 ps, 20 mm',  1*ps,  20*mm),
]

fig, axes = plt.subplots(len(cases), 2, figsize=(14, 4*len(cases)), tight_layout=True)

for row, (label, tau_val, L_val) in enumerate(cases):
    p = pulses.sech_pulse(t, tau_val, f_ref=f_ref, f0=c/lam_p,
                          Energy=1*uJ, Npwr_dB=200, frep=frep)
    s = pulses.sech_pulse(t, tau_val, f_ref=f_ref, f0=c/lam_s,
                          Energy=1*nJ, Npwr_dB=200, frep=frep)
    inp = p + s
    nee = dict(t=t, x=inp.a, Omega=Omega, f0=f_ref,
               L=L_val, D=D, b0=float(beta_ref), b1_ref=float(beta_1_ref),
               k=k_func, verbose=False)
    if BACKEND == 'jax':
        a, steps = nlo_jax.NEE(**nee, **nee_kwargs)
    else:
        a, steps = nlo_scipy.NEE(**nee)
    out = pulses.pulse(t, a, c/f_ref, frep)

    s_out = out.apply_filter(c/lam_s, 20*THz)
    i_out = out.apply_filter(c/lam_i, 20*THz)
    gain = 10*np.log10(s_out.energy_td() / s.energy_td())
    walkoff = abs(gvm_ps) * L_val/mm

    # Spectrum
    ax = axes[row, 0]
    out.plot_PSD(ax=ax, f_unit='um')
    inp.plot_PSD(ax=ax, f_unit='um')
    ax.get_lines()[-1].set(color='k', linestyle='--', alpha=0.3, label='Input')
    ax.get_lines()[-2].set(color='b', linewidth=1.5, label='Output')
    ax.set_xlim(0.9, 4.2)
    ax.legend(fontsize=10)
    ax.set_title(f'{label} — gain={gain:.0f}dB, walkoff={walkoff:.1f}ps')

    # Temporal
    ax = axes[row, 1]
    out.plot_magsq(ax=ax, t_unit='ps')
    inp.plot_magsq(ax=ax, t_unit='ps')
    ax.get_lines()[-1].set(color='k', linestyle='--', alpha=0.3, label='Input')
    ax.get_lines()[-2].set(color='b', linewidth=1.5, label='Output')
    tw = max(3*tau_val/ps, 15)
    ax.set_xlim(-tw, tw)
    ax.legend(fontsize=10); ax.set_title(f'{label} — temporal')

    print(f'{label}: gain={gain:.1f}dB, {len(steps)} steps, '
          f'E_idler={i_out.energy_td()/nJ:.3f}nJ')

plt.show()

## 5. Temperature tuning

Sweep crystal temperature to tune the idler wavelength across
the mid-IR. The QPM condition shifts with temperature because
the refractive indices change.

In [ ]:
T_values = np.arange(100, 210, 10)  # 100C to 200C
idler_centers = []
sig_gains = []

print(f'{"T (C)":>6s} | {"Gain (dB)":>10s} | {"E_idler (nJ)":>12s}')
print('-'*35)

for T_val in T_values:
    # Recompute dispersion at this temperature
    n_grid_T = np.ones(N)
    n_grid_T[valid] = np.array([materials.refractive_index('LN_MgO_e_T', w/um, T=T_val)
                                for w in wl_grid[valid]])
    beta_T = 2*pi*f_abs * n_grid_T / c
    n_s_T = materials.refractive_index('LN_MgO_e_T', lam_s/um, T=T_val)
    ng_s_T = group_index(lam_s/um, T_val)
    v_ref_T = c / ng_s_T
    beta_ref_T = 2*pi*f_ref * n_s_T / c
    D_T = beta_T - beta_ref_T - Omega/v_ref_T

    # Coupling stays the same (same d_eff, same poling)
    nee = dict(t=t, x=input_opa.a, Omega=Omega, f0=f_ref,
               L=L_crystal, D=D_T, b0=float(beta_ref_T),
               b1_ref=float(1/v_ref_T), k=k_func, verbose=False)

    if BACKEND == 'jax':
        a, _ = nlo_jax.NEE(**nee, **nee_kwargs)
    else:
        a, _ = nlo_scipy.NEE(**nee)
    out = pulses.pulse(t, a, c/f_ref, frep)

    s_out = out.apply_filter(c/lam_s, 20*THz)
    # Find idler by looking for peak in the 2.5-4.5 um range
    spec = np.abs(fft(a))**2
    wl_plot = c/f_abs
    idler_mask = (wl_plot > 2.5*um) & (wl_plot < 4.5*um) & (f_abs > 0)
    if np.any(spec[idler_mask] > 0):
        idx_peak = np.argmax(spec * idler_mask)
        lam_idler_meas = wl_plot[idx_peak]
        idler_centers.append(lam_idler_meas/um)
    else:
        idler_centers.append(float('nan'))

    i_out = out.apply_filter(c/lam_i, 20*THz)
    g = 10*np.log10(max(s_out.energy_td() / seed.energy_td(), 1e-10))
    sig_gains.append(g)

    print(f'{T_val:5.0f}C | {g:>9.1f} dB | {i_out.energy_td()/nJ:>10.3f} nJ')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), tight_layout=True)
ax1.plot(T_values, idler_centers, 'ro-')
ax1.set_xlabel('Temperature (C)'); ax1.set_ylabel('Idler wavelength (um)')
ax1.set_title('Idler Tuning with Temperature'); ax1.grid(True)

ax2.plot(T_values, sig_gains, 'bo-')
ax2.set_xlabel('Temperature (C)'); ax2.set_ylabel('Signal gain (dB)')
ax2.set_title('Gain vs Temperature'); ax2.grid(True)
plt.show()

## Summary

**Contrast with Tutorial 7 (degenerate OPO):**

| Property | Degenerate OPO (Tutorial 7) | Non-degenerate OPA (this) |
|----------|---------------------------|---------------------------|
| Pump | 1 um, 100 fs | 1.064 um, 10 ps |
| Signal | ~2 um (same as idler) | 1.507 um |
| Idler | ~2 um (degenerate) | 3.62 um (mid-IR) |
| Output spectrum | Octave-spanning, broadband | Narrow, well-defined peaks |
| Temporal profile | Wild modulation, sub-structure | Clean, smooth pulses |
| Application | Frequency combs, squeezed light | Spectroscopy, mid-IR generation |

Same underlying physics (chi(2) parametric interaction in PPLN),
radically different output character depending on configuration.

`BACKEND = 'jax'` is essential here — the N=2^14 grid makes CPU
propagation take ~15 minutes per pass, vs ~5-10s on GPU.